# Notebook 4 — Root-Cause Models Comparison (Corrected)

## Purpose
This notebook compares two supervised baseline models for the current Diagnostic Agent benchmark:

- **XGBoost**
- **Random Forest**

## Current benchmark target
At this stage, the supervised training benchmark includes only:
- `RC_CAPACITY_OVERLOAD`
- `RC_TRANSPORT_DELAY`

This corrected version:
- removes risky / leaky features
- adds **train / validation / test** metrics
- uses **Plotly** for presentation-ready graphs
- adds a **generalization-gap** analysis
- uses **XGBoost early stopping**

In [1]:
import os
import gc
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

VERBOSE = True
SAVE_ARTIFACTS = True
EXPORT_PLOTS = True
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def log(msg, level="INFO"):
    if VERBOSE:
        print(f"[{level}] {msg}")

def banner(title):
    if VERBOSE:
        print("\n" + "=" * 90)
        print(title)
        print("=" * 90)

def timed_run(name, fn, *args, **kwargs):
    start = time.time()
    log(f"START: {name}")
    result = fn(*args, **kwargs)
    elapsed = time.time() - start
    log(f"END: {name} | elapsed = {elapsed:.2f} sec")
    return result

banner("NOTEBOOK 4 INITIALIZATION")
log("Notebook 4 corrected version initialized")


NOTEBOOK 4 INITIALIZATION
[INFO] Notebook 4 corrected version initialized


In [2]:
banner("INSTALL / IMPORT VISUALIZATION PACKAGES")

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception:
    import sys
    !{sys.executable} -m pip install -q plotly
    import plotly.express as px
    import plotly.graph_objects as go

try:
    import kaleido  # noqa: F401
    log("kaleido already available")
except Exception:
    log("Installing kaleido for Plotly PNG export...", "WARN")
    import sys
    !{sys.executable} -m pip install -q kaleido
    import kaleido  # noqa: F401
    log("kaleido installed successfully")


INSTALL / IMPORT VISUALIZATION PACKAGES
[WARN] Installing kaleido for Plotly PNG export...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.3 MB/s eta 0:00:00
[INFO] kaleido installed successfully


In [3]:
banner("INSTALL / IMPORT MODELING PACKAGES")

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)

import joblib

try:
    from xgboost import XGBClassifier
except Exception:
    log("xgboost not found. Installing now...", "WARN")
    import sys
    !{sys.executable} -m pip install -q xgboost
    from xgboost import XGBClassifier
    log("xgboost installed successfully")


INSTALL / IMPORT MODELING PACKAGES


## Load Notebook 3 artifacts

We load:
- train / validation / test tabular splits
- feature column list

In [4]:
banner("LOAD NOTEBOOK 3 ARTIFACTS")

train_df = pd.read_parquet("/kaggle/input/datasets/azizamriiiiiiiiii/train-data1/benchmark_2class_train.parquet")
val_df   = pd.read_parquet("/kaggle/input/datasets/azizamriiiiiiiiii/train-data1/benchmark_2class_val.parquet")
test_df  = pd.read_parquet("/kaggle/input/datasets/azizamriiiiiiiiii/train-data1/benchmark_2class_test.parquet")

with open("/kaggle/input/datasets/azizamriiiiiiiiii/dddddddd/benchmark_feature_columns.json", "r") as f:
    feature_cols = json.load(f)

log(f"train_df shape = {train_df.shape}")
log(f"val_df shape   = {val_df.shape}")
log(f"test_df shape  = {test_df.shape}")
log(f"Number of original features = {len(feature_cols)}")

display(train_df.head(2))


LOAD NOTEBOOK 3 ARTIFACTS
[INFO] train_df shape = (421, 158)
[INFO] val_df shape   = (73, 158)
[INFO] test_df shape  = (83, 158)
[INFO] Number of original features = 118


,timestamp,zone_id,cell_id,node_id,device_type,latency_ms,jitter_ms,packet_loss_pct,throughput_mbps,bandwidth_util_pct,...,throughput_mbps_delta_5,bler_pct_model_rollmean_5,bler_pct_model_rollstd_5,bler_pct_model_delta_5,queue_length_rollmean_5,queue_length_rollstd_5,queue_length_delta_5,bandwidth_util_pct_rollmean_5,bandwidth_util_pct_rollstd_5,bandwidth_util_pct_delta_5
0,2026-03-27 17:22:08.968812,Z1,C1,N1,workstation,46.25,17.666666,0.0,0.01,0.066667,...,NaN,0.0,NaN,NaN,23.128000,NaN,NaN,0.066667,NaN,NaN
1,2026-03-27 17:34:37.461783,Z1,C1,N1,workstation,44.50,6.333333,0.0,0.03,0.200000,...,NaN,0.0,0.0,NaN,22.693501,0.614475,NaN,0.133333,0.094281,NaN


## Remove risky / leaky features

In [5]:
banner("REMOVE RISKY / LEAKY FEATURES")

RISKY_FEATURES = {
    "anomaly_flag",
    "anomaly_score",
    "anomaly_rate_recent",
    "hour_anomaly_rate",
    "baseline_phase",
    "traffic_confidence",
    "skip_for_training",
    "incident_recovery_time",
    "collection_completion_pct",
    "data_completeness_pct",
    "required_metrics_pct",
    "router_metrics_pct",
    "hour_of_day",
    "is_peak_hour",
}

original_feature_cols = feature_cols.copy()
feature_cols = [c for c in feature_cols if c not in RISKY_FEATURES]
dropped_features = [c for c in original_feature_cols if c in RISKY_FEATURES]

log(f"Original feature count = {len(original_feature_cols)}")
log(f"Filtered feature count = {len(feature_cols)}")
print("Dropped risky features:")
print(dropped_features)

with open("nb4_filtered_feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

log("Saved nb4_filtered_feature_columns.json")


REMOVE RISKY / LEAKY FEATURES
[INFO] Original feature count = 118
[INFO] Filtered feature count = 104
Dropped risky features:
['anomaly_flag', 'anomaly_rate_recent', 'anomaly_score', 'baseline_phase', 'collection_completion_pct', 'data_completeness_pct', 'hour_anomaly_rate', 'hour_of_day', 'incident_recovery_time', 'is_peak_hour', 'required_metrics_pct', 'router_metrics_pct', 'skip_for_training', 'traffic_confidence']
[INFO] Saved nb4_filtered_feature_columns.json


## Build the modeling matrices

In [6]:
banner("BUILD MODELING MATRICES")

TARGET_COL = "root_cause_label"

X_train = train_df[feature_cols].copy()
X_val   = val_df[feature_cols].copy()
X_test  = test_df[feature_cols].copy()

y_train_raw = train_df[TARGET_COL].copy()
y_val_raw   = val_df[TARGET_COL].copy()
y_test_raw  = test_df[TARGET_COL].copy()

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(y_test_raw)

class_names = list(le.classes_)
log(f"Class names = {class_names}")

positive_class_name = class_names[1] if len(class_names) == 2 else class_names[0]
log(f"Positive class for ROC/PR = {positive_class_name}")


BUILD MODELING MATRICES
[INFO] Class names = ['RC_CAPACITY_OVERLOAD', 'RC_TRANSPORT_DELAY']
[INFO] Positive class for ROC/PR = RC_TRANSPORT_DELAY


## Plot class balance

In [7]:
banner("PLOT CLASS BALANCE")

def save_plot(fig, name):
    if not EXPORT_PLOTS:
        return
    fig.write_html(f"{name}.html")
    try:
        fig.write_image(f"{name}.png", scale=2)
        log(f"Saved {name}.html and {name}.png")
    except Exception as e:
        log(f"Saved {name}.html | PNG export skipped ({e})", "WARN")

class_balance_df = pd.concat([
    train_df.assign(split="train")[[TARGET_COL, "split"]],
    val_df.assign(split="val")[[TARGET_COL, "split"]],
    test_df.assign(split="test")[[TARGET_COL, "split"]],
], axis=0)

class_balance_plot_df = (
    class_balance_df.groupby(["split", TARGET_COL])
    .size()
    .reset_index(name="rows")
)

fig_class_balance = px.bar(
    class_balance_plot_df,
    x="split",
    y="rows",
    color=TARGET_COL,
    barmode="group",
    title="Class Balance by Split",
    text="rows"
)
fig_class_balance.update_layout(template="plotly_white")
fig_class_balance.show()

save_plot(fig_class_balance, "nb4_class_balance_by_split")


PLOT CLASS BALANCE


[WARN] Saved nb4_class_balance_by_split.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Define evaluation helpers

In [8]:
banner("DEFINE EVALUATION HELPERS")

def compute_metrics(y_true, y_pred, y_prob, split_name, model_name):
    metrics = {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_prob is not None and y_prob.ndim == 2 and y_prob.shape[1] == 2:
        pos_prob = y_prob[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_true, pos_prob)
        metrics["pr_auc"] = average_precision_score(y_true, pos_prob)
    else:
        metrics["roc_auc"] = np.nan
        metrics["pr_auc"] = np.nan
    return metrics

def plot_confusion_matrix(cm, labels, title):
    fig = px.imshow(
        cm,
        x=labels,
        y=labels,
        text_auto=True,
        color_continuous_scale="Blues",
        title=title
    )
    fig.update_layout(template="plotly_white", xaxis_title="Predicted", yaxis_title="Actual")
    return fig


DEFINE EVALUATION HELPERS


## Train Random Forest

In [9]:
banner("TRAIN RANDOM FOREST")

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=350,
        max_depth=8,
        min_samples_split=4,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=RANDOM_SEED,
        class_weight="balanced",
        verbose=1
    ))
])

rf_train_start = time.time()
rf_pipeline.fit(X_train, y_train)
rf_train_time = time.time() - rf_train_start

log(f"Random Forest training time = {rf_train_time:.2f} sec")


TRAIN RANDOM FOREST


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 192 tasks      | elapsed:    0.3s


[INFO] Random Forest training time = 0.66 sec


[Parallel(n_jobs=-1)]: Done 350 out of 350 | elapsed:    0.6s finished


## Train XGBoost

In [10]:
banner("TRAIN XGBOOST")

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_SEED,
    tree_method="hist",
    early_stopping_rounds=30
)

xgb_train_start = time.time()
xgb_model.fit(
    X_train_imp,
    y_train,
    eval_set=[(X_val_imp, y_val)],
    verbose=25
)
xgb_train_time = time.time() - xgb_train_start

log(f"XGBoost training time = {xgb_train_time:.2f} sec")
log(f"Best iteration = {getattr(xgb_model, 'best_iteration', None)}")


TRAIN XGBOOST
[0]	validation_0-logloss:0.67577
[25]	validation_0-logloss:0.21090
[50]	validation_0-logloss:0.13752
[75]	validation_0-logloss:0.12691
[100]	validation_0-logloss:0.12943
[101]	validation_0-logloss:0.12938
[INFO] XGBoost training time = 0.30 sec
[INFO] Best iteration = 71


## Generate predictions

In [11]:
banner("GENERATE PREDICTIONS")

rf_train_pred = rf_pipeline.predict(X_train)
rf_val_pred   = rf_pipeline.predict(X_val)
rf_test_pred  = rf_pipeline.predict(X_test)

rf_train_prob = rf_pipeline.predict_proba(X_train)
rf_val_prob   = rf_pipeline.predict_proba(X_val)
rf_test_prob  = rf_pipeline.predict_proba(X_test)

xgb_train_pred = xgb_model.predict(X_train_imp)
xgb_val_pred   = xgb_model.predict(X_val_imp)
xgb_test_pred  = xgb_model.predict(X_test_imp)

xgb_train_prob = xgb_model.predict_proba(X_train_imp)
xgb_val_prob   = xgb_model.predict_proba(X_val_imp)
xgb_test_prob  = xgb_model.predict_proba(X_test_imp)

log("Train / validation / test predictions generated successfully")


GENERATE PREDICTIONS


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 350 out of 350 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 350 out of 350 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 350 out of 350 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 192 tasks      |

[INFO] Train / validation / test predictions generated successfully


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 350 out of 350 | elapsed:    0.1s finished


## Evaluate both models

In [12]:
banner("COMPUTE METRICS")

metrics_rows = []
metrics_rows.append(compute_metrics(y_train, rf_train_pred, rf_train_prob, "train", "Random Forest"))
metrics_rows.append(compute_metrics(y_val, rf_val_pred, rf_val_prob, "val", "Random Forest"))
metrics_rows.append(compute_metrics(y_test, rf_test_pred, rf_test_prob, "test", "Random Forest"))
metrics_rows.append(compute_metrics(y_train, xgb_train_pred, xgb_train_prob, "train", "XGBoost"))
metrics_rows.append(compute_metrics(y_val, xgb_val_pred, xgb_val_prob, "val", "XGBoost"))
metrics_rows.append(compute_metrics(y_test, xgb_test_pred, xgb_test_prob, "test", "XGBoost"))

metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df)

metrics_df.to_csv("nb4_model_metrics_patched.csv", index=False)
log("Saved nb4_model_metrics_patched.csv")


COMPUTE METRICS


,model,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,roc_auc,pr_auc
0,Random Forest,train,0.995249,0.994894,0.994894,0.995249,0.994894,0.994894,0.999903,0.999833
1,Random Forest,val,0.972603,0.972222,0.972556,0.972572,0.974359,0.972222,0.987988,0.989815
2,Random Forest,test,0.987952,0.988095,0.987952,0.987952,0.988095,0.988095,0.988966,0.992584
3,XGBoost,train,0.992874,0.993015,0.992352,0.992879,0.991703,0.993015,0.999709,0.999531
4,XGBoost,val,0.972603,0.972222,0.972556,0.972572,0.974359,0.972222,0.980480,0.987387
5,XGBoost,test,0.987952,0.988095,0.987952,0.987952,0.988095,0.988095,0.987224,0.991815


[INFO] Saved nb4_model_metrics_patched.csv


## Plot metric comparison

In [13]:
banner("PLOT METRIC COMPARISON")

plot_metrics = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "roc_auc", "pr_auc"]

metrics_long = metrics_df[metrics_df["split"].isin(["val", "test"])].melt(
    id_vars=["model", "split"],
    value_vars=plot_metrics,
    var_name="metric",
    value_name="value"
)

fig_metrics = px.bar(
    metrics_long,
    x="metric",
    y="value",
    color="model",
    barmode="group",
    facet_col="split",
    title="Model Comparison Across Validation and Test Metrics",
    text="value"
)
fig_metrics.update_traces(texttemplate="%{text:.3f}")
fig_metrics.update_layout(template="plotly_white")
fig_metrics.show()

save_plot(fig_metrics, "nb4_model_metrics_comparison")


PLOT METRIC COMPARISON


[WARN] Saved nb4_model_metrics_comparison.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Plot generalization gap

In [14]:
banner("PLOT GENERALIZATION GAP")

gap_metrics = ["accuracy", "macro_f1", "roc_auc", "pr_auc"]

gap_df = metrics_df.copy()
gap_long = gap_df.melt(
    id_vars=["model", "split"],
    value_vars=gap_metrics,
    var_name="metric",
    value_name="value"
)

fig_gap = px.line(
    gap_long,
    x="split",
    y="value",
    color="model",
    line_dash="metric",
    markers=True,
    title="Generalization Gap Across Train / Validation / Test"
)
fig_gap.update_layout(template="plotly_white")
fig_gap.show()

save_plot(fig_gap, "nb4_generalization_gap")


PLOT GENERALIZATION GAP


[WARN] Saved nb4_generalization_gap.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Confusion matrices

In [15]:
banner("CONFUSION MATRICES")

rf_cm_test = confusion_matrix(y_test, rf_test_pred)
xgb_cm_test = confusion_matrix(y_test, xgb_test_pred)

fig_rf_cm = plot_confusion_matrix(rf_cm_test, class_names, "Random Forest — Test Confusion Matrix")
fig_xgb_cm = plot_confusion_matrix(xgb_cm_test, class_names, "XGBoost — Test Confusion Matrix")

fig_rf_cm.show()
fig_xgb_cm.show()

save_plot(fig_rf_cm, "nb4_rf_test_confusion_matrix")
save_plot(fig_xgb_cm, "nb4_xgb_test_confusion_matrix")


CONFUSION MATRICES


[WARN] Saved nb4_rf_test_confusion_matrix.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)
[WARN] Saved nb4_xgb_test_confusion_matrix.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## ROC curves

In [16]:
banner("ROC CURVES")

rf_test_pos_prob = rf_test_prob[:, 1]
xgb_test_pos_prob = xgb_test_prob[:, 1]

rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_test_pos_prob)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_test_pos_prob)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=rf_fpr, y=rf_tpr, mode="lines", name="Random Forest"))
fig_roc.add_trace(go.Scatter(x=xgb_fpr, y=xgb_tpr, mode="lines", name="XGBoost"))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Chance", line=dict(dash="dash")))
fig_roc.update_layout(
    title="ROC Curve — Test Set",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white"
)
fig_roc.show()

save_plot(fig_roc, "nb4_test_roc_curve")


ROC CURVES


[WARN] Saved nb4_test_roc_curve.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Precision-Recall curves

In [17]:
banner("PRECISION-RECALL CURVES")

rf_prec, rf_rec, _ = precision_recall_curve(y_test, rf_test_pos_prob)
xgb_prec, xgb_rec, _ = precision_recall_curve(y_test, xgb_test_pos_prob)

fig_pr = go.Figure()
fig_pr.add_trace(go.Scatter(x=rf_rec, y=rf_prec, mode="lines", name="Random Forest"))
fig_pr.add_trace(go.Scatter(x=xgb_rec, y=xgb_prec, mode="lines", name="XGBoost"))
fig_pr.update_layout(
    title=f"Precision-Recall Curve — Test Set (Positive class: {positive_class_name})",
    xaxis_title="Recall",
    yaxis_title="Precision",
    template="plotly_white"
)
fig_pr.show()

save_plot(fig_pr, "nb4_test_precision_recall_curve")


PRECISION-RECALL CURVES


[WARN] Saved nb4_test_precision_recall_curve.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Feature importance

In [18]:
banner("FEATURE IMPORTANCE")

rf_importances = rf_pipeline.named_steps["model"].feature_importances_
rf_imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_importances
}).sort_values("importance", ascending=False).head(20)

fig_rf_imp = px.bar(
    rf_imp_df.iloc[::-1],
    x="importance",
    y="feature",
    orientation="h",
    title="Random Forest — Top 20 Feature Importances"
)
fig_rf_imp.update_layout(template="plotly_white")
fig_rf_imp.show()
save_plot(fig_rf_imp, "nb4_rf_feature_importance")

xgb_importances = xgb_model.feature_importances_
xgb_imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb_importances
}).sort_values("importance", ascending=False).head(20)

fig_xgb_imp = px.bar(
    xgb_imp_df.iloc[::-1],
    x="importance",
    y="feature",
    orientation="h",
    title="XGBoost — Top 20 Feature Importances"
)
fig_xgb_imp.update_layout(template="plotly_white")
fig_xgb_imp.show()
save_plot(fig_xgb_imp, "nb4_xgb_feature_importance")

rf_imp_df.to_csv("nb4_rf_feature_importance_top20.csv", index=False)
xgb_imp_df.to_csv("nb4_xgb_feature_importance_top20.csv", index=False)
log("Saved feature importance CSVs")


FEATURE IMPORTANCE


[WARN] Saved nb4_rf_feature_importance.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


[WARN] Saved nb4_xgb_feature_importance.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)
[INFO] Saved feature importance CSVs


## Prediction confidence distributions

In [19]:
banner("PREDICTION CONFIDENCE DISTRIBUTIONS")

rf_conf = np.max(rf_test_prob, axis=1)
xgb_conf = np.max(xgb_test_prob, axis=1)

conf_df = pd.DataFrame({
    "confidence": np.concatenate([rf_conf, xgb_conf]),
    "model": (["Random Forest"] * len(rf_conf)) + (["XGBoost"] * len(xgb_conf))
})

fig_conf = px.histogram(
    conf_df,
    x="confidence",
    color="model",
    barmode="overlay",
    nbins=30,
    title="Prediction Confidence Distribution — Test Set"
)
fig_conf.update_layout(template="plotly_white")
fig_conf.show()

save_plot(fig_conf, "nb4_prediction_confidence_distribution")


PREDICTION CONFIDENCE DISTRIBUTIONS


[WARN] Saved nb4_prediction_confidence_distribution.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Select the winner

In [20]:
banner("SELECT BEST MODEL")

val_metrics = metrics_df[metrics_df["split"] == "val"].copy()
val_metrics = val_metrics.sort_values(
    by=["macro_f1", "roc_auc", "accuracy"],
    ascending=[False, False, False]
).reset_index(drop=True)

best_model_name = val_metrics.iloc[0]["model"]
log(f"Best model selected = {best_model_name}")

test_metrics_best = metrics_df[
    (metrics_df["split"] == "test") &
    (metrics_df["model"] == best_model_name)
].copy()

display(val_metrics)
display(test_metrics_best)


SELECT BEST MODEL
[INFO] Best model selected = Random Forest


,model,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,roc_auc,pr_auc
0,Random Forest,val,0.972603,0.972222,0.972556,0.972572,0.974359,0.972222,0.987988,0.989815
1,XGBoost,val,0.972603,0.972222,0.972556,0.972572,0.974359,0.972222,0.980480,0.987387


,model,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,roc_auc,pr_auc
2,Random Forest,test,0.987952,0.988095,0.987952,0.987952,0.988095,0.988095,0.988966,0.992584


## Save the winning model and summary artifacts

In [21]:
banner("SAVE MODEL ARTIFACTS")

if best_model_name == "Random Forest":
    best_model = rf_pipeline
else:
    best_model = {"imputer": imputer, "model": xgb_model}

joblib.dump(le, "nb4_label_encoder.joblib")

if best_model_name == "Random Forest":
    joblib.dump(best_model, "nb4_best_model_random_forest_patched.joblib")
else:
    joblib.dump(best_model, "nb4_best_model_xgboost_patched.joblib")

summary = {
    "best_model": best_model_name,
    "target_classes": class_names,
    "positive_class_for_roc_pr": positive_class_name,
    "rf_train_time_sec": round(rf_train_time, 3),
    "xgb_train_time_sec": round(xgb_train_time, 3),
    "n_features_after_patch": len(feature_cols),
    "dropped_risky_features": dropped_features,
    "all_metrics": metrics_df.to_dict(orient="records")
}

with open("nb4_model_selection_summary_patched.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

log("Saved patched model artifacts and summary")


SAVE MODEL ARTIFACTS
[INFO] Saved patched model artifacts and summary
